In [16]:
import pandas as pd


In [17]:
df_setup = pd.read_csv("/Users/anthonylin/Desktop/Project/Taiwan-company-survival/data/raw/company_setup_201501.csv.csv", dtype=str)

print(df_setup.shape)
print(df_setup.columns)
print(df_setup.head())

(3449, 7)
Index(['序號', '統一編號', '公司名稱', '公司所在地', '代表人', '資本額', '核准設立日期'], dtype='object')
  序號      統一編號         公司名稱              公司所在地  代表人      資本額   核准設立日期
0  1  24675546     高裕貿易有限公司     高雄市鹽埕區大勇路27號1樓  吳姿慧  1000000  1031230
1  2  24675617  蟹皇宴餐飲事業有限公司      高雄市苓雅區青年二路67號  葉家榮   500000  1031231
2  3  24675644   岡山鐵馬企業有限公司     高雄市岡山區平和路31號1樓  邱志華  1000000  1031231
3  4  24675671     穩固企業有限公司  高雄市鳳山區新康街16巷17號1樓  柳緣浩   500000  1031231
4  5  24675686     德美貿易有限公司    高雄市岡山區通校路170號1樓   郭昊   300000  1031231


In [18]:
print(df_setup["核准設立日期"].min())
print(df_setup["核准設立日期"].max())
print(df_setup["公司所在地"].str[:3].value_counts())

1031217
1040130
公司所在地
臺北市    947
新北市    606
臺中市    511
高雄市    337
桃園市    315
臺南市    198
彰化縣    123
新竹市     71
新竹縣     68
苗栗縣     37
屏東縣     35
雲林縣     34
宜蘭縣     33
基隆市     29
花蓮縣     23
嘉義縣     23
嘉義市     22
南投縣     15
臺東縣      9
金門縣      9
澎湖縣      4
Name: count, dtype: int64


In [19]:
df_dissolve = pd.read_csv("/Users/anthonylin/Desktop/Project/Taiwan-company-survival/data/raw/company_dissolve_201501.csv", dtype=str)

print(df_dissolve.shape)
print(df_dissolve.columns)
print(df_dissolve.head())

(1513, 8)
Index(['序號', '統一編號', '公司名稱', '公司所在地', '代表人', '資本額', '核准設立日期', '核准解散日期'], dtype='object')
  序號      統一編號        公司名稱                      公司所在地  代表人       資本額   核准設立日期  \
0  1  01517594    立凱企業有限公司            新北市新店區安祥路一０六之七號  NaN   7000000  0741229   
1  2  03152402      正龍有限公司      臺北市松山區復興北路141巷6弄17號1樓  NaN   2000000  0520413   
2  3  03373313  偉進企業股份有限公司            臺北市萬華區忠孝西路2段34號  NaN   5000000  0610221   
3  4  04233163    鑫立企業有限公司  臺北市大安區忠孝東路3段217巷3弄17-1號2樓  NaN  15000000  0631106   
4  5  04493239  隆宇貿易股份有限公司             臺北市萬華區內江街70號1樓  NaN   1000000  0690409   

    核准解散日期  
0  1040112  
1  1040107  
2  1040105  
3  1040123  
4  1040106  


In [20]:
import requests

In [21]:
# ===== 設立清冊 =====
response = requests.get("https://data.gov.tw/api/v2/rest/dataset/6047")
data = response.json()

rows = []
for item in data["result"]["distribution"]:
    rows.append({
        "month": item["resourceDescription"],
        "url": item["resourceDownloadUrl"],
    })

setup_list = pd.DataFrame(rows)
setup_list.to_csv("../data/raw/file_list_setup.csv", index=False)
print("設立清冊：", len(setup_list), "個月")
print(setup_list.head())

設立清冊： 164 個月
      month                                                url
0  2013年02月  https://data.gcis.nat.gov.tw/od/file?oid=63F2D...
1  2013年03月  https://data.gcis.nat.gov.tw/od/file?oid=B2E74...
2  2013年04月  https://data.gcis.nat.gov.tw/od/file?oid=6C311...
3  2013年05月  https://data.gcis.nat.gov.tw/od/file?oid=6405B...
4  2013年06月  https://data.gcis.nat.gov.tw/od/file?oid=EF471...


In [22]:
# ===== 解散清冊 =====
response = requests.get("https://data.gov.tw/api/v2/rest/dataset/6049")
data = response.json()

rows = []
for item in data["result"]["distribution"]:
    rows.append({
        "month": item["resourceDescription"],
        "url": item["resourceDownloadUrl"],
    })

dissolve_list = pd.DataFrame(rows)
dissolve_list.to_csv("../data/raw/file_list_dissolve.csv", index=False)
print("解散清冊：", len(dissolve_list), "個月")
print(dissolve_list.head())

解散清冊： 164 個月
      month                                                url
0  2013年02月  https://data.gcis.nat.gov.tw/od/file?oid=C27A4...
1  2013年03月  https://data.gcis.nat.gov.tw/od/file?oid=07AE0...
2  2013年04月  https://data.gcis.nat.gov.tw/od/file?oid=91D00...
3  2013年05月  https://data.gcis.nat.gov.tw/od/file?oid=BACC1...
4  2013年06月  https://data.gcis.nat.gov.tw/od/file?oid=1C503...


In [23]:
import os
import time
import requests
import pandas as pd
from tqdm import tqdm

for kind in ["setup", "dissolve"]:

    # 建立存放資料夾，例如 data/raw/setup/
    folder = f"../data/raw/{kind}"
    os.makedirs(folder, exist_ok=True)

    # 讀取上一步存好的目錄
    file_list = pd.read_csv(f"../data/raw/file_list_{kind}.csv")

    for _, row in tqdm(file_list.iterrows(), total=len(file_list), desc=kind):

        # "2015年01月" → "201501"
        month = row["month"].replace("年", "").replace("月", "")
        save_path = f"{folder}/company_{kind}_{month}.csv"

        # 已經下載過就跳過
        if os.path.exists(save_path):
            continue

        # 下載並存檔
        response = requests.get(row["url"])
        with open(save_path, "wb") as f:
            f.write(response.content)

        # 休息 1 秒，避免對政府網站造成負擔
        time.sleep(1)

print("下載完成")

dissolve: 100%|██████████| 164/164 [06:26<00:00,  2.35s/it]

下載完成
